# 04 - SVM method

**Aim:** implement the practical SVM protocol for signal-peptide detection, with feature extraction and parameter selection inside nested cross-validation.

We used the practical instructions in `04 - Methods - SVM.pdf` to plan this section. We will reuse the representative proteins and folds from 02b, build composition and hydrophobicity features, and select the SVM settings using training data only. The analysis in 02c provides biological context, and 03 provides our loading and reporting conventions.

**Materials**
- `04 - Methods - SVM.pdf` (slides 22–26 and 30–41; practical instructions on slide 41).
- `04_feature_extraction_hydrophobicity.ipynb` (setup, A.1–A.5 and B.4).
- `04_SVM-sklearn_revised.ipynb` (sections 1–3 and 7).
- `03_von_heijne_method.ipynb` (loading in 1.1; metrics and mean ± SE in 5.4–5.6).

**Status:** we prepared this notebook as a work guide and left the code cells empty. All implementation, execution, tables, figures and observations are pending.

## Work distribution and handoffs

We defined five consecutive blocks so each person can complete their contribution and pass the cumulative notebook to the next. Person 1 handles shared preparation; Person 5 handles final integration. Everyone can read the materials in advance, but implementation follows **Person 1 → Person 2 → Person 3 → Person 4 → Person 5**.

| Person | Scientific sections | Receives | Passes on |
| --- | --- | --- | --- |
| 1 | 0–1: data and common definitions | Prepared FASTA/TSV and practical materials. | To Person 2: `data`, reserved `benchmarking_data`, paths, fixed hydropathy settings and the training-only `get_n` helper. |
| 2 | 2: feature extraction | Person 1's data and definitions. | To Person 3: composition/profile functions, `extract_features(proteins, n)`, feature names and catalog. |
| 3 | 3: RF ranking and SVM helpers | Person 2's extractors and feature order. | To Person 4: ranking, coupled parameter-grid, scaler/SVM and evaluation helpers. |
| 4 | 4–5: inner selection and outer evaluation | Person 3's helpers and all preceding contributions. | To Person 5: five external-fold metrics, predictions, selected settings/features and inner-search scores. |
| 5 | 6: reporting and final integration | Person 4's results and the cumulative notebook. | To the group: integrated notebook, result tables, figures and observations. |

We kept names and interfaces common across the blocks:

| Object | Interface |
| --- | --- |
| `data` | Training proteins with `accession`, `sequence`, `label`, `split`, `cv_fold` and original metadata. |
| `get_n(train_data)` | Integer N-terminal length derived only from positive SP lengths in the supplied training subset. |
| `extract_features(proteins, n)` | Numeric DataFrame indexed by accession, with the same ordered feature columns for every subset. |
| RF ranking | Column indices ordered by importance, plus names and importances from the current training fit. |
| Search candidates | Valid tuples `(k, C, gamma)`, with gamma calculated for that candidate's k. |
| `inner_scores` | `test_fold`, `validation_fold`, `n`, `k`, `C`, `gamma`, `mcc`. |
| `cv_per_round` | `test_fold`, `n`, `selected_k`, `C`, `gamma`, `precision`, `recall`, `accuracy`, `mcc`, `f1`. |
| `cv_predictions` | `accession`, `test_fold`, `label`, `prediction`. |
| `selected_features` | `test_fold`, `feature`, `rank`, `importance`, after the outer-training refit. |

All fitted quantities belong to their current training subset. The handoffs contain definitions and reusable functions first, then actual results after Person 4 completes CV. Each person adds the explanation for their own block before handing it on.

## 0. Data loading

We kept paths relative to `notebooks/` and selected the Conda `LB2` environment and the same kernel as 03.

### 0.1 Imports and paths

**Pending — Person 1.** Prepare imports for sequence handling, numerical tables, plotting, RF, scaling, SVC and metrics. Define `input_dir = ../data/prepared`, `table_dir = ../results/tables/04` and `figure_dir = ../results/figures/04`. Create output directories when implementation starts.

**Deliverable:** imports and shared path variables. If using Biopython for the scale or profile, add it to `requirements.txt` and install it in Conda LB2 during implementation; the example's Colab pip cell is an environment example.

**Similar code:** imports in `04_SVM-sklearn_revised.ipynb`, section 0, and the hydrophobicity notebook's setup. Relative paths and output conventions follow 03 section 0.

### 0.2 Load the prepared proteins

**Pending — Person 1.** Load `positive_dataset.tsv`, `negative_dataset.tsv` and their matching FASTA files. Map sequences by accession and assign `label = 1` for SP and `0` for NO_SP. Retain the saved metadata and nullable-integer `cv_fold`. Separate the training table `data` from `benchmarking_data`.

**Deliverable:** 8,146 training proteins (881 SP / 7,265 NO_SP), with folds 1–5, and a separate reserved benchmarking table. Keep the existing partition unchanged.

**Similar code:** reuse `read_fasta` and the accession mapping from 03 section 1.1. The SVM example section 1 loads breast-cancer data and uses a random split; that loader/split does not apply to this project.

## 1. Common feature definitions

We separated fixed choices from quantities learned in each training subset. The composition length n will be learned from positive training proteins; it is not a fixed 90-residue window (slides 30 and 36–40).

### 1.1 Estimate n from the current training subset

**Pending — Person 1.** Define `get_n(train_data)` using the median `cleavage_pos` of its positive proteins. In our prepared data, `cleavage_pos` is the last SP residue, so it also gives the annotated SP length. The function receives only the training subset supplied by the caller.

The slides do not specify how to convert a half-integer median to a sequence length. Use the ceiling of the median and document this project convention. The caller will recompute n for each three-fold inner-training set and again for the four-fold outer-training refit. Apply that same n to the corresponding validation/test sequences.

**Deliverable:** the helper definition and its rounding convention; no global n or data-derived fixed composition matrix. `cleavage_pos` helps estimate n from training positives but is not a predictor.

**Similar code:** no equivalent median-length helper in the two practical notebooks. Slide 40 specifies where to compute n; 02c establishes the meaning of `cleavage_pos`.

### 1.2 Fixed hydropathy settings and residue policy

**Pending — Person 1.** Document the Kyte–Doolittle scale and define `hydropathy_length = 40` and `hydropathy_window = 5`, following the example in slides 32–34. These are starting feature choices, not universal requirements. Use the available region for shorter sequences.

We chose complete windows, with uniform weights and no padding. For unknown residues, preserve their positions and average the known values within each original window; a window with no known values is missing. Composition likewise excludes non-canonical residues from its denominator. Summaries will use the non-missing profile values. Do not silently delete residues before forming windows or replace unknowns with zero.

**Deliverable:** scale, fixed settings and the shared policy for Person 2. The 40-residue hydropathy segment is separate from the composition segment of length n. Keep these settings fixed during the prescribed k/C/gamma search.

**Similar code:** hydrophobicity notebook setup and A.1–A.3 cover the scale and complete windows. A.5 supplies explicit known-residue averaging, but its final `hydrophobicity_profile` shrinks windows at the ends; adapt that unknown-residue logic to our complete-window choice.

## 2. Feature extraction

We chose a starting set of 22 features from slides 30–34: 20 amino-acid fractions and two summaries of a hydropathy profile. Additional features are encouraged in the practical material, but charge, volume and other scales are optional extensions rather than assigned requirements here.

### 2.1 N-terminal amino-acid composition

**Pending — Person 2; receives sections 0–1 from Person 1.** Define a composition function receiving a sequence and n. Count canonical residues in the available first n positions and return 20 fractions in alphabet order `ACDEFGHIKLMNPQRSTVWY`. Use column names `aa_fraction_A` through `aa_fraction_Y`.

**Deliverable:** a per-protein extractor whose length comes from its n argument. It must not estimate n itself or use that protein's own cleavage annotation to choose its region.

**Similar code:** the practical notebooks do not include this composition function. Adapt canonical counting from 02c section 5.2 from pooled class counts to individual proteins; slide 30 defines the required vector.

### 2.2 Hydropathy profile and summaries

**Pending — Person 2.** Define an explicit complete-window profile using the scale and settings from 1.2. From the first 40 residues, calculate maximum and mean over its non-missing 5-position window averages. Name the outputs `hydropathy_max_w5` and `hydropathy_profile_mean_w5`.

**Deliverable:** profile and aggregation functions. The mean feature is the mean of the window profile, not the direct mean of all residue-scale values. For a canonical sequence of length L, the complete-window profile contains L − w + 1 values.

**Similar code:** `window_profile` in hydrophobicity notebook A.2 is the closest loop; A.3 compares it with `ProteinAnalysis.protein_scale`. A.5 explains the unknown-residue issue; B.4 shows profile-to-feature aggregation. Slide 34 gives the max/mean pair used here.

### 2.3 Assemble the feature extractor and catalog

**Pending — Person 2.** Define `extract_features(proteins, n)` returning an accession-indexed DataFrame: 20 composition columns followed by the two hydropathy columns. Keep the feature order identical for all subsets. Prepare a catalog with `feature`, `definition`, `region`, `scale`, `window`, `aggregation` and `unit`.

**Deliverable to Person 3:** the extraction functions, 22 feature names and catalog. Composition depends on the supplied n; hydropathy uses the fixed settings in 1.2. Keep labels, folds, lengths, organism, `cleavage_pos` and `has_tm_in_90` outside the predictor matrix. Extra features, if later added, need a documented definition and an updated feature count D.

**Similar code:** hydrophobicity notebook B.4 assembles a small feature dictionary, but does not build our protein matrix. Slide 36 shows how these features enter the model. No global composition matrix should be reused across CV splits.

## 3. RF selection and SVM helpers

We followed the order in slides 35–36: extract features, rank them with RF, keep top-k, standardize selected columns, and train an RBF SVM. Person 3 prepares the helpers; Person 4 will call them inside CV.

### 3.1 Fit an RF ranking

**Pending — Person 3; receives Person 2's extractor and feature order.** Define a helper that fits RF on the supplied training matrix and labels, returning ranked column indices, feature names and importances. Use fixed settings of 500 trees and `random_state = 0`, as in slide 35.

**Deliverable:** one ranking per current training set. Reuse that ranking for every candidate k, C and gamma. With the initial grid, the five outer rounds require 20 inner-training rankings and five outer-training rankings; do not fit a new forest for every parameter tuple.

**Similar code:** slide 35 provides the fit and ranking operations. The theoretical worked example has `select_rf`, but refits a forest for each k; use slide 35's reuse strategy for this practical.

### 3.2 Build the coupled parameter grid

**Pending — Person 3.** Use D from the extractor's actual feature count. Build unique valid k values from ceiling(D/4), ceiling(D/2) and D; with the starting D = 22, these are 6, 11 and 22. Use C values 1, 4 and 16. For each k, use gamma values 0.5/k, 1/k and 2/k.

**Deliverable:** 27 initial `(k, C, gamma)` tuples for the starting feature set, in a documented deterministic order. Generate gamma for each k; it is not one independent numerical gamma list shared by every k. Define exact ties to retain the first tuple, with smaller k considered first.

**Similar code:** slide 38 gives the grid; slide 40 shows its use. `04_SVM-sklearn_revised.ipynb` explains gamma in sections 4.2 and 6 but does not implement this joint search.

### 3.3 Standardize selected features and fit SVC

**Pending — Person 3.** Define helpers receiving selected training columns, training labels and a candidate C/gamma. Fit `StandardScaler` on those training columns and fit an RBF SVC on the transformed values. Prediction must apply the same fitted scaler to validation/test columns in the same order.

**Deliverable:** a fitted scaler/SVC pair, or a simple scaler/SVC Pipeline, for Person 4's candidate evaluation and outer refit. Leave probability calibration and additional model settings outside this practical workflow.

**Similar code:** SVM notebook section 3 provides scaling and SVC fitting; section 7 shows a Pipeline keeping the scaler and SVC together. Its fixed parameters and random dataset split are demonstrations; use the candidates and saved-fold subsets from this guide.

### 3.4 Calculate classification metrics

**Pending — Person 3.** Prepare a helper receiving true labels and predictions, returning MCC, precision, recall, F1 and accuracy with the same names as 03. Positive class 1 means SP. Use MCC alone to score candidates; the full metric set will describe external-fold performance.

**Deliverable to Person 4:** the ranking, grid, fitting/prediction and metric helpers. The next block receives definitions rather than a model trained on all proteins.

**Similar code:** `evaluate` in SVM notebook section 1 calculates these five metrics. Adapt its printed output to a result record; 03 section 5 supplies the project's table naming conventions.

## 4. Inner model selection

We chose explicit loops from slide 40 so the changing composition length and reused RF ranking are visible. Selection will use the four saved folds available within each outer-training set, with three for training and one for validation.

### 4.1 Select outer and inner subsets

**Pending — Person 4; receives the completed helpers from Person 3.** Define the five outer rounds from saved `cv_fold` identifiers 1–5. For each round, hold one fold out as test and use the other four as outer training. Within those four, choose each in turn as inner validation and use the other three for inner training.

**Deliverable:** subset selection for the nested loops. The slides label subsets 0–4; our existing 1–5 labels identify the same five-fold structure and remain unchanged. Benchmarking is excluded.

**Similar code:** slides 37 and 40 describe these subsets. Neither practical notebook implements the full nested protocol. The cyclic `get_cv_split` from 03 needs replacement because it uses one validation fold per outer round.

### 4.2 Prepare each inner-training fit

**Pending — Person 4.** For each inner split, compute n from its three-fold training positives using `get_n`. Call `extract_features` on inner training and validation with that same n. Fit one RF ranking using inner-training features and labels.

**Deliverable:** training/validation matrices, aligned labels, n and the ranking for the current inner split. These can be retained within the current outer round while scoring the grid. The next inner split must estimate its own n and ranking.

**Similar code:** slide 40 gives this exact sequence. The feature functions from the hydrophobicity notebook support extraction, but the training-dependent composition length is our adaptation to the practical instructions.

### 4.3 Score the grid and choose a combination

**Pending — Person 4.** For each `(k, C, gamma)` candidate, take the first k ranked columns, fit the scaler/SVM on inner training, predict inner validation and record MCC in `inner_scores`. Average each candidate's MCC across the four validation folds and choose the highest mean, using the tie convention in 3.2.

Before the outer refit, inspect whether the optimum lies on a searchable grid boundary. Extend the relevant candidate range using only inner-validation results, as required by slide 38, and record the final grid used. k cannot exceed D; selecting all D columns is a dimensional limit. Reuse the appropriate inner-training matrices and rankings when adding candidates.

**Deliverable:** best k/C/gamma for the current outer round and the candidate scores. Inner maxima are selection scores, not performance estimates. All choices must be complete before predicting the outer test fold.

**Similar code:** slides 38–40 are the primary reference. The practical SVM notebook does not include this search; it deliberately fixes parameters to explain SVC behavior.

## 5. Outer refit and evaluation

We planned one external test per saved fold, after all inner choices have been made. The procedure will be repeated five times, without repartitioning the data (slides 37–41).

### 5.1 Refit on the four outer-training folds

**Pending — Person 4.** Recompute n using all positive proteins in the current four-fold outer-training set. Extract outer-training and outer-test features with that n. Fit a new RF ranking on outer training, select its best k columns, and refit the scaler/SVM with the chosen C and gamma.

**Deliverable:** the final fitted procedure for this round, its n, selected feature names and their ranks/importances. Do not reuse one inner split's n, ranking or model as the final refit.

**Similar code:** the refit block in slide 40 specifies every step. SVM notebook section 3 supports scaler/SVC fitting, and hydrophobicity notebook A/B.4 supports profile extraction and aggregation.

### 5.2 Predict and record all five external folds

**Pending — Person 4.** Predict the held-out outer test proteins once with the refitted procedure and calculate the five test metrics. Repeat sections 4–5 for all saved test folds. Preserve accessions and labels in `cv_predictions`, and record n, selected k/C/gamma and metrics in `cv_per_round`. Record the outer-refit selection in `selected_features`.

**Deliverable to Person 5:** five result rows, test predictions for all 8,146 training proteins once each, selected-feature records and `inner_scores`. These counts describe the intended coverage, not executed results. Do not tune against external predictions or evaluate benchmarking here.

**Similar code:** slides 39–41 define the outputs. SVM notebook section 1 provides `evaluate`; its single holdout demonstration is not the five-round project protocol.

## 6. Results and integration

We followed 03's reporting conventions and the practical requirement to report each metric as mean ± standard error across the five external folds. All tables, figures and observations below are pending.

### 6.1 Per-round results and mean ± SE

**Pending — Person 5; receives Person 4's completed CV results.** Display `cv_per_round` and summarize precision, recall, accuracy, MCC and F1 across the five rounds. Use the sample standard deviation (`ddof = 1`) and SE = SD / sqrt(5), matching 03. Keep summary columns `mean`, `std` and `stderr`, indexed by metric.

**Deliverable:** `table_dir / 6_1_cv_per_round.tsv`, `table_dir / 6_1_cv_summary.tsv` and `table_dir / 6_1_cv_predictions.tsv`. Save full precision and round only for display; label reported uncertainty as SE.

**Similar code:** 03 sections 5.4–5.5 already save per-round tables and calculate mean ± SE. The practical SVM notebook section 1 supplies individual metrics; slides 39–41 require the five-round summary.

### 6.2 Selected settings and feature records

**Pending — Person 5.** Show the selected n, k, C and gamma for each outer round, together with feature names and their outer-training importances. Export `selected_features`, `inner_scores` and the feature catalog. Describe variation across training sets without treating one ranked list as a biological finding.

**Deliverable:** `table_dir / 6_2_selected_features.tsv`, `table_dir / 6_2_inner_scores.tsv` and `table_dir / 6_2_feature_catalog.tsv`. The catalog must distinguish variable first-n composition from the fixed 40-residue hydropathy segment.

**Similar code:** slide 39 illustrates the settings table; slide 35 exposes importances. The two practical notebooks do not export these project tables. Follow the TSV convention in 03 section 5.

### 6.3 Plot external-fold metrics

**Pending — Person 5.** Plot the five metric means with SE and individual external-fold values, using consistent labels and colors. Save the figure as `figure_dir / 6_3_cv_metrics.pdf`, print its path and display it.

**Deliverable:** the external-fold metric figure. The points and error bars must come from the result tables; do not use illustrative values from slide 39 or outputs from either example dataset.

**Similar code:** 03 section 5.6 provides the mean ± SE plot. The practical notebooks show other plots, but not this five-round summary.

### 6.4 Observations and final integration

**Pending — Person 5.** Write brief observations about SP detection, parameter variation and selected features, using the actual external results. Read `../results/tables/03/5_4_cv_per_round.tsv` and `../results/tables/03/5_5_cv_summary.tsv` only for a descriptive comparison with the completed Von Heijne method; its training protocol differs from the SVM protocol.

Bring the five contributions together, retain the scientific order, and replace completed task instructions with explanations of what we actually did. Update README status/results when supported by real implementation and checks. Keep benchmarking reserved, and leave any unfinished tasks marked pending.

**Deliverable:** the integrated notebook and a short results-based discussion. Model internals, probability calibration, repeated random splits and all toy exercises are supporting examples rather than extra assigned analyses.

**Similar code:** the interpretation in 03 section 5.10 is a style reference. The practical examples explain their own proteins/datasets, so their observations cannot serve as our results.